## JSON data — the format you'll see constantly (APIs, web data)

In [145]:
obj = '''
{
  "name":"wes",
  "places_lived":["Perth","Adelaide","Sydney"],
  "pet":null,
  "siblings":[
    {"name":"Scott","age":30,"pets":["Zeus","Zuko"]},
    {"name":"Katie","age":38,"pets":["Sixes","Stache","Cisco"]}
  ]
}
'''
obj

'\n{\n  "name":"wes",\n  "places_lived":["Perth","Adelaide","Sydney"],\n  "pet":null,\n  "siblings":[\n    {"name":"Scott","age":30,"pets":["Zeus","Zuko"]},\n    {"name":"Katie","age":38,"pets":["Sixes","Stache","Cisco"]}\n  ]\n}\n'

## Convert JSON string → Python object

In [146]:
import json
result = json.loads(obj)
result

{'name': 'wes',
 'places_lived': ['Perth', 'Adelaide', 'Sydney'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 30, 'pets': ['Zeus', 'Zuko']},
  {'name': 'Katie', 'age': 38, 'pets': ['Sixes', 'Stache', 'Cisco']}]}

### Convert Python object → JSON string

In [147]:
asjson = json.dumps(result)
type(asjson)

str

The real workflow: JSON → DataFrame

In [148]:
import pandas as pd
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age'])
siblings

,name,age
0,Scott,30
1,Katie,38


## Pandas `.read_json()` Function automatically convert JSON datasets into Series or DataFrame . 

In [149]:

import pandas as pd


with open("examples/example8.json", "w") as f:
    f.write("""
[
    {"a": 1, "b": 2, "c": 3},
    {"a": 4, "b": 5, "c": 6},
    {"a": 7, "b": 8, "c": 9}
]
""")

jobj = pd.read_json("examples/example8.json")
print(jobj)

   a  b  c
0  1  2  3
1  4  5  6
2  7  8  9


In [150]:
print(jobj.to_json())
print(jobj.to_json(orient='records'))

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}
[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]


## read_json needs a "table-shaped" JSON (list of flat dicts). to_json() default is column-oriented — use orient='records' to get back a row-list shape like typical API JSON.

# BASIC MEANING OF  WEBSCRAPING :

# 📌 Introduction to Web Scraping

## 🔎 What is Web Scraping?
Web scraping means:
- Downloading a webpage
- Extracting useful information from its HTML

---

## 🖥️ Example Webpage
Suppose a webpage contains:

```html
<div class="product">
    <h2>HP Victus</h2>
    <span>65000</span>
</div>
```

## 📤 Extracted Data
A scraper extracts:

Laptop Name:`HP Victus`

Price    : `65000`

<h2>--> then we can load it into pandas 

In [151]:
df = pd.DataFrame({
    "Laptop": ["HP Victus"],
    "Price": [65000]
})

### Pandas provides read_html(), which uses libraries such as lxml and BeautifulSoup behind the scenes to extract HTML tables directly into DataFrames.

## workflow:
Website<br>
      ↓<br>
HTML<br>
      ↓<br>
read_html()<br>
      ↓<br>
DataFrame<br>
      ↓<br>
Data Analysis<br>

In [152]:
html_content = """
<html>
<head><title>Failed Bank List</title></head>
<body>
<h1>FDIC Failed Bank List</h1>
<table>
  <tr><th>Bank Name</th><th>City</th><th>ST</th><th>CERT</th><th>Closing Date</th></tr>
  <tr><td>The Woodbury Banking Company</td><td>Woodbury</td><td>GA</td><td>11297</td><td>August 19, 2016</td></tr>
  <tr><td>First CornerStone Bank</td><td>King of Prussia</td><td>PA</td><td>35312</td><td>May 6, 2016</td></tr>
  <tr><td>Trust Company Bank</td><td>Memphis</td><td>TN</td><td>9956</td><td>April 29, 2016</td></tr>
</table>
</body>
</html>
"""

with open('examples/fdic_failed_bank_list.html', 'w') as f:
    f.write(html_content)

In [153]:
tables=pd.read_html("examples/fdic_failed_bank_list.html")
tables

[                      Bank Name             City  ST   CERT     Closing Date
 0  The Woodbury Banking Company         Woodbury  GA  11297  August 19, 2016
 1        First CornerStone Bank  King of Prussia  PA  35312      May 6, 2016
 2            Trust Company Bank          Memphis  TN   9956   April 29, 2016]

## It searches every table element in the HTML page and returns:
## list[dataFrame]

In [154]:
print(len(tables))
failure=tables[0]
failure.head()

1


,Bank Name,City,ST,CERT,Closing Date
0,The Woodbury Banking Company,Woodbury,GA,11297,"August 19, 2016"
1,First CornerStone Bank,King of Prussia,PA,35312,"May 6, 2016"
2,Trust Company Bank,Memphis,TN,9956,"April 29, 2016"


In [155]:
close_timestamps = pd.to_datetime(failure['Closing Date'])
close_timestamps.dt.year.value_counts()

Closing Date
2016    3
Name: count, dtype: int64

### pd.to_datetime() converts that whole column into real datetime

- .dt is a special accessor pandas gives you on datetime columns 
-  it unlocks date-specific properties like .dt.year, .dt.month, .dt.day.

In [156]:
from lxml import objectify 
path ='examples/Performance_MNR.xml'
parsed=objectify.parse(open(path))
root=parsed.getroot()

XML File<br>
    ↓<br>
objectify.parse()<br>
    ↓<br>
Parsed XML Tree<br>
    ↓<br>
root = Root Element<br>
### root is now a special lxml object representing `PERFORMANCE`, the top of the tree.








In [157]:
data = []
skip_fields = ['PARENT_SEQ', 'INDICATOR_SEQ', 'DESIRED_CHANGE', 'DECIMAL_PLACES']

for elt in root.INDICATOR:
    el_data = {}
    for child in elt.getchildren():
        if child.tag in skip_fields:
            continue
        el_data[child.tag] = child.pyval
    data.append(el_data)

perf = pd.DataFrame(data)
perf

AttributeError: no such child: INDICATOR

In [ ]:
el_data = {}

for child in root:
    el_data[child.tag] = child.text

print(el_data)

for child in root:
    print(child.tag, child.text)


{'INDICATOR': None}
INDICATOR None


### The purpose of parsing XML is to extract the values stored inside XML tags and convert them into a tabular format (DataFrame) for analysis

In [ ]:
from lxml import objectify
parsed=objectify.parse("examples/final_xml.xml")
root=parsed.getroot()
print(root.tag)


COLLEGE


In [ ]:
for student in root.STUDENT :
    print(student.NAME)

Dipak
Rahul


In [ ]:
data = []

for student in root.STUDENT:

    row = {}

    row["ROLL_NO"] = student.ROLL_NO.text
    row["NAME"] = student.NAME.text
    row["BRANCH"] = student.BRANCH.text
    row["SEMESTER"] = student.SEMESTER.text
    row["CGPA"] = student.CGPA.text
    row["EMAIL"] = student.EMAIL.text
    row["PHONE"] = student.PHONE.text

    row["CITY"] = student.ADDRESS.CITY.text
    row["STATE"] = student.ADDRESS.STATE.text
    row["PIN"] = student.ADDRESS.PIN.text

    data.append(row)

In [ ]:
df=pd.DataFrame(data)
df

""


In [ ]:
## HTML link tag , which is also valid XML.

from io import StringIO
tag='<a href="https://www.google.com">Google</a>'
root=objectify.parse(StringIO(tag)).getroot()
root

<Element a at 0x27d12822a80>

In [ ]:
root.get('href')

'http://www.google.com'